In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl"

Mounted at /content/drive
Processing ./drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl


In [2]:
import pandas as pd
import src.config as config
import os
# Entrenamiento (Noviembre)
from src.modelling.transforms import EngagementScaler
from src.modelling.similarity import KendallTauSimilarity
from src.modelling.spectral import SpectralClusteringModel

In [3]:
config.WORKPATH = "drive/MyDrive/Master Data Science/TFM/GITHUB"

In [5]:
cols = config.FEATURES_TIME_BASED
workpath = config.WORKPATH
n_clusters = config.CLUSTER_TIME_BASED


df_train = pd.read_parquet(os.path.join(workpath, "processed", "training_201911", "aggregated_time_based.parquet"))
df_test = pd.read_parquet(os.path.join(workpath, "processed", "validation_201912", "aggregated_time_based.parquet"))
# 1. Ranking y normalización
scaler = EngagementScaler(features=cols)
df_ranked = scaler.fit(df_train)  # Calcula means y stds
df_normalized = scaler.transform(df_ranked)
df_normalized['cluster'] = None  # placeholder

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix = similarity.fit_transform(df_normalized, features=cols)

# 3. Clustering y guardar artefactos
spectral_model = SpectralClusteringModel(n_clusters=config.CLUSTER_TIME_BASED, n_init=50)
labels = spectral_model.fit(affinity_matrix, df_normalized, features=cols)

# Guardar estadísticas de entrenamiento
spectral_model.set_training_statistics(df_ranked, features=cols)

# Obtener resultados
df_clusters = spectral_model.get_clusters_dataframe(df_normalized)

# Guardar artefactos
spectral_model.save_model_artifacts(os.path.join(workpath, "models", "nov_time_based_artifacts.np"))


# ============= PROYECCIÓN (Diciembre) =============

# 1. Cargar estadísticas de entrenamiento
scaler_dec = EngagementScaler(features=cols)
scaler_dec.means = spectral_model.train_rank_means
scaler_dec.stds = spectral_model.train_rank_stds

# 2. Aplicar transformación (sin reajustar)
df_ranked_dec = df_test.copy()
df_ranked_dec = scaler_dec.rank(df_ranked_dec)
df_normalized_dec = scaler_dec.transform(df_ranked_dec)

# 3. Proyectar en espacio de centroides
df_normalized_dec["cluster"] = spectral_model.predict(df_normalized_dec[cols].values)

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix_dec = similarity.fit_transform(df_normalized_dec, features=cols)

Computing label assignment using kmeans
Initialization complete
Iteration 0, inertia 0.01665170849580652.
Iteration 1, inertia 0.010782934806743794.
Iteration 2, inertia 0.010592711365680017.
Iteration 3, inertia 0.010584649475153076.
Converged at iteration 3: strict convergence.
Initialization complete
Iteration 0, inertia 0.017631842804271717.
Iteration 1, inertia 0.013291675564812042.
Iteration 2, inertia 0.010723561302998551.
Iteration 3, inertia 0.01059598548977133.
Iteration 4, inertia 0.010564886992711279.
Iteration 5, inertia 0.010522864156239983.
Converged at iteration 5: strict convergence.
Initialization complete
Iteration 0, inertia 0.016214850150968345.
Iteration 1, inertia 0.010905551505148577.
Iteration 2, inertia 0.010691055946853895.
Iteration 3, inertia 0.010543680230485065.
Iteration 4, inertia 0.01052476697441902.
Converged at iteration 4: strict convergence.
Initialization complete
Iteration 0, inertia 0.013282638561200542.
Iteration 1, inertia 0.011762321473596998

In [6]:
affinity_matrix_dec.to_parquet(os.path.join(workpath, "processed", "validation_201912", "affinity_matrix_time_based.parquet"))
affinity_matrix.to_parquet(os.path.join(workpath, "processed", "training_201911", "affinity_matrix_time_based.parquet"))